In [3]:
# MOMISH intervention plots — SDR_output_Aug_11_9am
# 4 counties, implementation-index scenarios (base/current/moderate/high)
# includes combined ("All") package scenarios
# adapted from MOMISH_Plots5.ipynb


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

In [5]:
# =========================================================
# USER SETTINGS
# =========================================================

RUN_INTERVENTIONS = {
    "mentors": True,
    "prompts": True,
    "fqa": True,
    "pulse": True,
    "referral": True,
    "blood": True,
    "combined": True,
}

BASE_DIR = Path()

DATA_ROOT = BASE_DIR 
PLOTS_ROOT = BASE_DIR

COUNTY_FOLDER_SLUGS = {
    "Kakamega": "kakamega",
    "Kisii": "kisii",
    "Mombasa": "mombasa",
    "Makueni": "makueni",
}

COUNTIES = list(COUNTY_FOLDER_SLUGS.keys())

COUNTY_DIRS = {
    county: DATA_ROOT / f"scenario_comparison_results_{slug}"
    for county, slug in COUNTY_FOLDER_SLUGS.items()
}

CANONICAL_SCENARIO_FILES = {
    "scenario_definitions.csv",
    "scenario_run_results.csv",
    "scenario_location_run_results.csv",
    "scenario_mean_results.csv",
}

def resolve_county_csv(county_dir: Path, filename: str) -> Path:
    """Resolve current canonical scenario_comparison.py output files only."""
    if filename not in CANONICAL_SCENARIO_FILES:
        raise ValueError(f"Unexpected scenario output filename: {filename}")
    path = county_dir / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {filename} under {county_dir}")
    return path

SCENARIO_PREFIX = {
    "mentors": "mentors",
    "prompts": "prompts",
    "fqa": "fqa",
    "pulse": "pulse",
    "referral": "referral",
    "blood": "blood",
    "combined": "combined",
}

scenario_definitions_df = pd.concat(
    [
        pd.read_csv(resolve_county_csv(county_dir, "scenario_definitions.csv")).assign(County=county)
        for county, county_dir in COUNTY_DIRS.items()
    ],
    ignore_index=True,
)
scenario_definitions_df.columns = scenario_definitions_df.columns.str.strip()


def intervention_name_from_metadata(file_key):
    prefix = SCENARIO_PREFIX[file_key]
    names = (
        scenario_definitions_df.loc[
            scenario_definitions_df["scenario"].str.startswith(f"{prefix}_"),
            "intervention",
        ]
        .dropna()
        .unique()
    )
    if len(names) != 1:
        raise ValueError(f"Expected one intervention name for {file_key}; found {list(names)}")
    return names[0]

STANDARD_LEVELS = ["base", "current", "moderate", "high"]

LEVEL_TO_LABEL = {
    "base": "Baseline",
    "current": "Current",
    "moderate": "Moderate",
    "high": "High",
}

CHANGE_LEVELS = ["current", "moderate", "high"]
CHANGE_LEVEL_LABELS = [LEVEL_TO_LABEL[l] for l in CHANGE_LEVELS]

SCENARIOS = [LEVEL_TO_LABEL[l] for l in STANDARD_LEVELS]

EXCLUDE_FIRST_N_RUNS = 0
MIN_INCLUDED_RUN = EXCLUDE_FIRST_N_RUNS + 1

BASE_OUTCOMES = [
    {
        "summary_name": "Maternal mortality ratio",
        "outcome_col": "maternal_deaths_per_100k",
        "multiplier": 1,
        "ylabel": "MMR per 100,000 live births",
        "plot_name": "maternal_mortality_ratio",
        "lower_is_better": True,
    },
    {
        "summary_name": "Maternal complications",
        "outcome_col": "severe_complication",
        "multiplier": 100,
        "ylabel": "Maternal complications (%)",
        "plot_name": "maternal_complications",
        "lower_is_better": True,
    },
]

INTERVENTION_CONFIGS = {
    "mentors": {
        "intervention_name": intervention_name_from_metadata("mentors"),
        "outcomes_to_plot": BASE_OUTCOMES,
    },
    "prompts": {
        "intervention_name": intervention_name_from_metadata("prompts"),
        "outcomes_to_plot": BASE_OUTCOMES + [
            {
                "summary_name": "Antenatal care rate",
                "outcome_col": "anc",
                "multiplier": 100,
                "ylabel": "Antenatal care rate (%)",
                "plot_name": "antenatal_care_rate",
                "lower_is_better": False,
            },
            {
                "summary_name": "Normal referral",
                "outcome_col": "self_referral",
                "multiplier": 100,
                "ylabel": "Normal referral (%)",
                "plot_name": "normal_referral",
                "lower_is_better": False,
            },
        ],
        "location_distribution_plots": [
            {
                "rate_col": "initial_l45",
                "plot_title": "Initial delivery at L4/5",
                "plot_name": "initial_delivery_l45_rate",
            },
            {
                "rate_col": "final_l45",
                "plot_title": "Final delivery at L4/5",
                "plot_name": "final_delivery_l45_rate",
            },
        ],
    },
    "fqa": {
        "intervention_name": intervention_name_from_metadata("fqa"),
        "outcomes_to_plot": BASE_OUTCOMES,
    },
    "pulse": {
        "intervention_name": intervention_name_from_metadata("pulse"),
        "outcomes_to_plot": BASE_OUTCOMES,
    },
    "referral": {
        "intervention_name": intervention_name_from_metadata("referral"),
        "outcomes_to_plot": BASE_OUTCOMES,
    },
    "blood": {
        "intervention_name": intervention_name_from_metadata("blood"),
        "outcomes_to_plot": BASE_OUTCOMES,
    },
    "combined": {
        "intervention_name": intervention_name_from_metadata("combined"),
        "outcomes_to_plot": BASE_OUTCOMES + [
            {
                "summary_name": "Antenatal care rate",
                "outcome_col": "anc",
                "multiplier": 100,
                "ylabel": "Antenatal care rate (%)",
                "plot_name": "antenatal_care_rate",
                "lower_is_better": False,
            },
            {
                "summary_name": "Normal referral",
                "outcome_col": "self_referral",
                "multiplier": 100,
                "ylabel": "Normal referral (%)",
                "plot_name": "normal_referral",
                "lower_is_better": False,
            },
        ],
        "location_distribution_plots": [
            {
                "rate_col": "initial_l45",
                "plot_title": "Initial delivery at L4/5",
                "plot_name": "initial_delivery_l45_rate",
            },
            {
                "rate_col": "final_l45",
                "plot_title": "Final delivery at L4/5",
                "plot_name": "final_delivery_l45_rate",
            },
        ],
    },
}

PCT_CHANGE_EXCLUDE_INTERVENTIONS = {
    "Maternal complications": [
        INTERVENTION_CONFIGS["referral"]["intervention_name"],
        INTERVENTION_CONFIGS["blood"]["intervention_name"],
    ],
}

run_df = pd.concat(
    [
        pd.read_csv(resolve_county_csv(county_dir, "scenario_run_results.csv")).assign(County=county)
        for county, county_dir in COUNTY_DIRS.items()
    ],
    ignore_index=True,
)
run_df.columns = run_df.columns.str.strip()
run_df = run_df[run_df["run"] >= MIN_INCLUDED_RUN].copy()

location_df = pd.concat(
    [
        pd.read_csv(resolve_county_csv(county_dir, "scenario_location_run_results.csv")).assign(County=county)
        for county, county_dir in COUNTY_DIRS.items()
    ],
    ignore_index=True,
)
location_df.columns = location_df.columns.str.strip()
location_df = location_df[location_df["run"] >= MIN_INCLUDED_RUN].copy()

print(f"Using runs {MIN_INCLUDED_RUN}+ after excluding the first {EXCLUDE_FIRST_N_RUNS} runs")
print(f"Loaded run-level rows: {len(run_df):,}")
print(f"Loaded location-level rows: {len(location_df):,}")
print(f"Counties: {COUNTIES}")
print(f"Scenarios per county: {run_df.groupby('County')['scenario'].nunique().to_dict()}")


Using runs 1+ after excluding the first 0 runs
Loaded run-level rows: 23,200
Loaded location-level rows: 92,800
Counties: ['Kakamega', 'Kisii', 'Mombasa', 'Makueni']
Scenarios per county: {'Kakamega': 29, 'Kisii': 29, 'Makueni': 29, 'Mombasa': 29}


In [6]:
# =========================================================
# STYLE
# =========================================================

SDR_COLORS = {
    "paper": "#F6F3ED",
    "paper_deep": "#EDE7DA",
    "card": "#FFFFFF",
    "ink": "#1C1A15",
    "ink_soft": "#4A4339",
    "ink_muted": "#7E7464",
    "border": "#E2DAC8",
    "accent": "#B5471F",
    "baseline": "#9C9082",
    "current": "#C4511E",
    "moderate": "#D4A056",
    "high": "#2E5F5C",
}

COUNTY_COLORS = {
    "Kakamega": "#2E5F5C",
    "Kisii": "#5C6BC0",
    "Mombasa": "#C4511E",
    "Makueni": "#1B6B8A",
}

SCENARIO_COLORS = {
    "Baseline": SDR_COLORS["baseline"],
    "Current": SDR_COLORS["current"],
    "Moderate": SDR_COLORS["moderate"],
    "High": SDR_COLORS["high"],
}

plt.rcParams.update({
    "figure.facecolor": SDR_COLORS["paper"],
    "axes.facecolor": SDR_COLORS["card"],
    "axes.edgecolor": SDR_COLORS["border"],
    "axes.labelcolor": SDR_COLORS["ink_soft"],
    "xtick.color": SDR_COLORS["ink_muted"],
    "ytick.color": SDR_COLORS["ink_muted"],
    "text.color": SDR_COLORS["ink"],
    "font.family": "sans-serif",
    "legend.frameon": False,
    "grid.color": SDR_COLORS["border"],
    "grid.linestyle": "--",
    "grid.linewidth": 0.7,
})


def _save_plot(fig, plot_dir, file_prefix, plot_name, timestamp):
    plot_dir.mkdir(parents=True, exist_ok=True)
    save_path = plot_dir / f"{file_prefix}_{plot_name}_{timestamp}.png"
    fig.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight",
        facecolor=SDR_COLORS["paper"],
    )
    plt.show()
    print(f"Saved plot to: {save_path}")


def _style_axes(ax):
    ax.grid(axis="y", alpha=0.55)
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(SDR_COLORS["border"])
    ax.spines["bottom"].set_color(SDR_COLORS["ink_muted"])
    ax.tick_params(axis="both", colors="black")


def _annotate_bars(ax, bars, values):
    for bar, value in zip(bars, values):
        if pd.notna(value):
            label = f"{value:,.0f}" if abs(value) >= 10 else f"{value:.1f}"
            ax.annotate(
                label,
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7.5,
                color=SDR_COLORS["ink"],
                clip_on=False,
                zorder=5,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    boxstyle="round,pad=0.12",
                    alpha=0.75,
                ),
            )


def available_scenario_labels(df, county=None, county_col="County", scenario_col="Scenario Label"):
    """Return scenario labels present in the generated data, preserving standard order."""
    if df is None or len(df) == 0 or scenario_col not in df.columns:
        return list(SCENARIOS)
    subset = df
    if county is not None and county_col in subset.columns:
        subset = subset[subset[county_col] == county]
    present = set(subset[scenario_col].dropna())
    labels = [label for label in SCENARIOS if label in present]
    return labels or list(SCENARIOS)


def scenarios_for_county(county, df=None):
    """Scenario labels available for this county in the generated data."""
    return available_scenario_labels(df, county=county)


def drop_unavailable_current(df, county_col="County", scenario_col="Scenario Label"):
    """Keep rows exactly as generated; missing levels are handled as NaN during plotting."""
    return df.copy() if df is not None else df


def get_standard_scenarios(prefix):
    return [f"{prefix}_{level}" for level in STANDARD_LEVELS]


def load_intervention_run_data(file_key, intervention_name):
    prefix = SCENARIO_PREFIX[file_key]
    subset = run_df[run_df["scenario"].str.startswith(f"{prefix}_")].copy()
    subset = subset[subset["level"].isin(STANDARD_LEVELS)].copy()

    if subset.empty:
        raise ValueError(f"No rows found for intervention '{intervention_name}'")

    subset["Scenario Label"] = subset["level"].map(LEVEL_TO_LABEL)
    subset["Intervention"] = subset["intervention"] if "intervention" in subset.columns else intervention_name
    subset["Run"] = subset["run"]
    return subset


def load_intervention_location_data(file_key, intervention_name):
    prefix = SCENARIO_PREFIX[file_key]
    subset = location_df[location_df["scenario"].str.startswith(f"{prefix}_")].copy()
    subset = subset[subset["level"].isin(STANDARD_LEVELS)].copy()

    if subset.empty:
        raise ValueError(f"No location rows found for intervention '{intervention_name}'")

    subset["Scenario Label"] = subset["level"].map(LEVEL_TO_LABEL)
    subset["Intervention"] = subset["intervention"] if "intervention" in subset.columns else intervention_name
    subset["Run"] = subset["run"]

    if "location" in subset.columns:
        subset["Delivery Location"] = subset["location"].astype(str).str.strip()
    else:
        raise ValueError("location column missing from scenario_location_run_results.csv")

    return subset


def summarize_preaggregated_outcome(df, outcome_col, multiplier, outcome_name, group_cols):
    if outcome_col not in df.columns:
        raise ValueError(f"Column not found for outcome: {outcome_col}")

    run_summary = df.copy()
    run_summary["value"] = run_summary[outcome_col] * multiplier

    summary = (
        run_summary.groupby(group_cols, observed=True)
        .agg(
            mean=("value", "mean"),
            sd=("value", "std"),
            n_runs=("Run", "nunique"),
        )
        .reset_index()
    )

    summary["ci95"] = 1.96 * summary["sd"] / np.sqrt(summary["n_runs"])
    summary["lower"] = summary["mean"] - summary["ci95"]
    summary["upper"] = summary["mean"] + summary["ci95"]
    summary["Outcome"] = outcome_name
    return summary


def make_change_table(summary_df, baseline_label="Baseline"):
    tables = []
    for county in COUNTIES:
        county_df = summary_df[summary_df["County"] == county].copy()
        baseline_rows = county_df.loc[county_df["Scenario Label"] == baseline_label, "mean"]
        if baseline_rows.empty:
            continue
        baseline_mean = baseline_rows.iloc[0]
        table = county_df.copy()
        table["Absolute Change vs Baseline"] = table["mean"] - baseline_mean
        table["Percent Change vs Baseline"] = (
            table["Absolute Change vs Baseline"] / baseline_mean * 100
        )
        table = table[table["Scenario Label"] != baseline_label].copy()
        tables.append(table)

    if not tables:
        return pd.DataFrame()

    out = pd.concat(tables, ignore_index=True)[
        [
            "County",
            "Intervention",
            "Scenario Label",
            "Outcome",
            "mean",
            "upper",
            "lower",
            "Absolute Change vs Baseline",
            "Percent Change vs Baseline",
        ]
    ].round(2)
    return drop_unavailable_current(out)


def plot_outcome_by_county(summary_df, outcome_name, ylabel, plot_name, intervention_name, file_prefix, plot_dir, timestamp, x_axis="scenario"):
    plot_df = summary_df[summary_df["Outcome"] == outcome_name].copy()

    fig, ax = plt.subplots(figsize=(13.5, 6.4))
    fig.patch.set_facecolor(SDR_COLORS["paper"])
    ax.set_facecolor(SDR_COLORS["card"])

    if x_axis == "county":
        # Counties on x-axis; scenarios in legend
        x = np.arange(len(COUNTIES))
        width = 0.18
        offsets = np.linspace(-(len(SCENARIOS) - 1) / 2, (len(SCENARIOS) - 1) / 2, len(SCENARIOS))

        for i, scenario in enumerate(SCENARIOS):
            scenario_df = (
                plot_df[plot_df["Scenario Label"] == scenario]
                .set_index("County")
                .reindex(COUNTIES)
            )
            xpos = x + offsets[i] * width
            bars = ax.bar(
                xpos,
                scenario_df["mean"],
                width,
                yerr=scenario_df["ci95"],
                capsize=2,
                label=scenario,
                color=SCENARIO_COLORS[scenario],
                edgecolor="none",
                alpha=0.95,
                error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
            )
            _annotate_bars(ax, bars, scenario_df["mean"])

        ax.set_xlabel("County", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(COUNTIES, fontsize=10)
        ax.legend(loc="upper right", fontsize=9, ncol=2, title="Scenario")
    else:
        # Scenarios on x-axis; counties in legend (default)
        x = np.arange(len(SCENARIOS))
        width = 0.18
        offsets = np.linspace(-(len(COUNTIES) - 1) / 2, (len(COUNTIES) - 1) / 2, len(COUNTIES))

        for i, county in enumerate(COUNTIES):
            county_df = (
                plot_df[plot_df["County"] == county]
                .set_index("Scenario Label")
                .reindex(SCENARIOS)
            )
            xpos = x + offsets[i] * width
            bars = ax.bar(
                xpos,
                county_df["mean"],
                width,
                yerr=county_df["ci95"],
                capsize=2,
                label=county,
                color=COUNTY_COLORS[county],
                edgecolor="none",
                alpha=0.95,
                error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
            )
            _annotate_bars(ax, bars, county_df["mean"])

        ax.set_xlabel("Scenario", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(SCENARIOS, fontsize=10)
        ax.legend(loc="upper right", fontsize=9, ncol=2)

    ax.set_title(
        f"{intervention_name}: {outcome_name} — county comparison",
        fontsize=18,
        pad=16,
        color=SDR_COLORS["ink"],
    )
    ax.set_ylabel(ylabel, fontsize=11)
    _style_axes(ax)
    ax.margins(y=0.22)
    plt.tight_layout()
    _save_plot(fig, plot_dir, file_prefix, plot_name, timestamp)


def plot_outcome_by_location(summary_df, outcome_name, ylabel, plot_name, intervention_name, county, file_prefix, plot_dir, timestamp):
    plot_df = summary_df[
        (summary_df["Outcome"] == outcome_name) & (summary_df["County"] == county)
    ].copy()

    location_order = ["Home", "L2/3", "L4", "L5"]
    locations = [loc for loc in location_order if loc in plot_df["Delivery Location"].unique()]
    scenarios = scenarios_for_county(county, plot_df)

    fig, ax = plt.subplots(figsize=(12.5, 6.2))
    fig.patch.set_facecolor(SDR_COLORS["paper"])
    ax.set_facecolor(SDR_COLORS["card"])

    x = np.arange(len(locations))
    width = 0.18
    offsets = np.linspace(-(len(scenarios) - 1) / 2, (len(scenarios) - 1) / 2, len(scenarios))

    for i, scenario in enumerate(scenarios):
        scenario_df = (
            plot_df[plot_df["Scenario Label"] == scenario]
            .set_index("Delivery Location")
            .reindex(locations)
        )
        xpos = x + offsets[i] * width
        bars = ax.bar(
            xpos,
            scenario_df["mean"],
            width,
            yerr=scenario_df["ci95"],
            capsize=2,
            label=scenario,
            color=SCENARIO_COLORS[scenario],
            edgecolor="none",
            alpha=0.95,
            error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
        )
        _annotate_bars(ax, bars, scenario_df["mean"])

    ax.set_title(
        f"{intervention_name}: {outcome_name} — {county}",
        fontsize=18,
        pad=16,
        color=SDR_COLORS["ink"],
    )
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("Delivery location", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(locations, fontsize=10.5)
    ax.legend(loc="upper right", fontsize=9)
    _style_axes(ax)
    ax.margins(y=0.22)
    plt.tight_layout()
    _save_plot(fig, plot_dir / county.lower(), file_prefix, plot_name, timestamp)


def plot_outcome_single_county(summary_df, outcome_name, ylabel, plot_name, intervention_name, county, file_prefix, plot_dir, timestamp):
    plot_df_source = summary_df[
        (summary_df["Outcome"] == outcome_name) & (summary_df["County"] == county)
    ].copy()
    scenarios = scenarios_for_county(county, plot_df_source)
    plot_df = (
        plot_df_source
        .set_index("Scenario Label")
        .reindex(scenarios)
    )

    fig, ax = plt.subplots(figsize=(9.0, 6.0))
    fig.patch.set_facecolor(SDR_COLORS["paper"])
    ax.set_facecolor(SDR_COLORS["card"])

    x = np.arange(len(scenarios))
    colors = [SCENARIO_COLORS[s] for s in scenarios]

    bars = ax.bar(
        x,
        plot_df["mean"],
        width=0.62,
        yerr=plot_df["ci95"],
        capsize=3,
        color=colors,
        edgecolor="none",
        alpha=0.95,
        error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
    )
    _annotate_bars(ax, bars, plot_df["mean"])

    ax.set_title(
        f"{intervention_name}: {outcome_name} — {county}",
        fontsize=18,
        pad=16,
        color=SDR_COLORS["ink"],
    )
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("Scenario", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios, fontsize=10.5)
    _style_axes(ax)
    ax.margins(y=0.22)
    plt.tight_layout()
    _save_plot(fig, plot_dir / county.lower(), file_prefix, plot_name, timestamp)


def plot_l45_by_county(summary_df, plot_title, plot_name, intervention_name, file_prefix, plot_dir, timestamp, x_axis="scenario"):
    plot_df = summary_df[summary_df["Outcome"] == plot_title].copy()

    fig, ax = plt.subplots(figsize=(13.0, 5.8))
    fig.patch.set_facecolor(SDR_COLORS["paper"])
    ax.set_facecolor(SDR_COLORS["card"])

    if x_axis == "county":
        x = np.arange(len(COUNTIES))
        width = 0.18
        offsets = np.linspace(-(len(SCENARIOS) - 1) / 2, (len(SCENARIOS) - 1) / 2, len(SCENARIOS))

        for i, scenario in enumerate(SCENARIOS):
            scenario_df = (
                plot_df[plot_df["Scenario Label"] == scenario]
                .set_index("County")
                .reindex(COUNTIES)
            )
            xpos = x + offsets[i] * width
            bars = ax.bar(
                xpos,
                scenario_df["mean"],
                width,
                yerr=scenario_df["ci95"],
                capsize=2,
                label=scenario,
                color=SCENARIO_COLORS[scenario],
                edgecolor="none",
                alpha=0.95,
                error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
            )
            for bar, value in zip(bars, scenario_df["mean"]):
                if pd.notna(value):
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + (scenario_df["ci95"].max() or 0) * 0.05,
                        f"{value:.0f}%",
                        ha="center",
                        va="bottom",
                        fontsize=7.5,
                        color=SDR_COLORS["ink"],
                    )

        ax.set_xlabel("County", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(COUNTIES, fontsize=10)
        ax.legend(loc="upper right", fontsize=9, ncol=2, title="Scenario")
    else:
        x = np.arange(len(SCENARIOS))
        width = 0.18
        offsets = np.linspace(-(len(COUNTIES) - 1) / 2, (len(COUNTIES) - 1) / 2, len(COUNTIES))

        for i, county in enumerate(COUNTIES):
            county_df = (
                plot_df[plot_df["County"] == county]
                .set_index("Scenario Label")
                .reindex(SCENARIOS)
            )
            xpos = x + offsets[i] * width
            bars = ax.bar(
                xpos,
                county_df["mean"],
                width,
                yerr=county_df["ci95"],
                capsize=2,
                label=county,
                color=COUNTY_COLORS[county],
                edgecolor="none",
                alpha=0.95,
                error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
            )
            for bar, value in zip(bars, county_df["mean"]):
                if pd.notna(value):
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + (county_df["ci95"].max() or 0) * 0.05,
                        f"{value:.0f}%",
                        ha="center",
                        va="bottom",
                        fontsize=7.5,
                        color=SDR_COLORS["ink"],
                    )

        ax.set_xlabel("Scenario", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(SCENARIOS, fontsize=10)
        ax.legend(loc="upper right", fontsize=9, ncol=2)

    ax.set_title(
        f"{intervention_name}: {plot_title} — county comparison",
        fontsize=18,
        pad=16,
        color=SDR_COLORS["ink"],
    )
    ax.set_ylabel("Share of mothers (%)", fontsize=11)
    _style_axes(ax)
    ax.margins(y=0.22)
    plt.tight_layout()
    _save_plot(fig, plot_dir, file_prefix, plot_name, timestamp)


In [7]:
def build_pct_change_grid(outcome_spec, exclude_interventions=None):
    exclude_interventions = exclude_interventions or []
    rows = []

    for file_key, enabled in RUN_INTERVENTIONS.items():
        if not enabled:
            continue

        intervention_name = INTERVENTION_CONFIGS[file_key]["intervention_name"]
        if intervention_name in exclude_interventions:
            continue

        prefix = SCENARIO_PREFIX[file_key]
        subset = load_intervention_run_data(file_key, intervention_name)

        for county in COUNTIES:
            county_df = subset[subset["County"] == county]
            baseline = (
                county_df.loc[county_df["Scenario Label"] == "Baseline", outcome_spec["outcome_col"]]
                .mean()
                * outcome_spec["multiplier"]
            )

            for level in CHANGE_LEVELS:
                label = LEVEL_TO_LABEL[level]
                scenario_mean = (
                    county_df.loc[county_df["Scenario Label"] == label, outcome_spec["outcome_col"]]
                    .mean()
                    * outcome_spec["multiplier"]
                )
                pct_change = (scenario_mean - baseline) / baseline * 100 if baseline else np.nan
                rows.append({
                    "Intervention": intervention_name,
                    "County": county,
                    "Scenario Label": label,
                    "Percent change vs baseline": pct_change,
                })

    return pd.DataFrame(rows)


def plot_county_pct_change_subplots(outcome_spec, exclude_interventions=None):
    grid_df = build_pct_change_grid(outcome_spec, exclude_interventions=exclude_interventions)
    if grid_df.empty:
        print(f"No data for {outcome_spec['summary_name']} pct-change grid")
        return grid_df

    interventions = [
        INTERVENTION_CONFIGS[k]["intervention_name"]
        for k, enabled in RUN_INTERVENTIONS.items()
        if enabled and INTERVENTION_CONFIGS[k]["intervention_name"] not in (exclude_interventions or [])
    ]

    n_rows = len(interventions)
    n_cols = len(COUNTIES)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.0 * n_cols + 1.2, 2.2 * n_rows + 1.0),
        sharex=True,
    )
    fig.patch.set_facecolor(SDR_COLORS["paper"])

    if n_rows == 1:
        axes = np.array([axes])
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    bar_colors = [SCENARIO_COLORS[label] for label in CHANGE_LEVEL_LABELS]

    all_values = grid_df["Percent change vs baseline"].abs().max()
    y_limit = max(5, np.ceil(all_values / 5) * 5) if pd.notna(all_values) else 10

    for r, intervention in enumerate(interventions):
        for c, county in enumerate(COUNTIES):
            ax = axes[r, c]
            ax.set_facecolor(SDR_COLORS["card"])

            cell = (
                grid_df[
                    (grid_df["Intervention"] == intervention)
                    & (grid_df["County"] == county)
                ]
                .set_index("Scenario Label")
                .reindex(CHANGE_LEVEL_LABELS)
            )

            values = cell["Percent change vs baseline"].to_numpy(dtype=float)
            x = np.arange(len(CHANGE_LEVEL_LABELS))

            bars = ax.bar(
                x,
                values,
                color=bar_colors,
                width=0.62,
                edgecolor="none",
                alpha=0.95,
            )

            for bar, value in zip(bars, values):
                if pd.notna(value):
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + np.sign(value) * 0.8,
                        f"{value:.1f}%",
                        ha="center",
                        va="bottom" if value >= 0 else "top",
                        fontsize=8,
                        color=SDR_COLORS["ink"],
                    )

            ax.axhline(0, color=SDR_COLORS["ink_muted"], linewidth=0.8)
            ax.set_ylim(-y_limit, y_limit)
            ax.set_xticks(x)
            ax.set_xticklabels(CHANGE_LEVEL_LABELS, fontsize=8, rotation=35, ha="right")
            ax.grid(axis="y", alpha=0.4)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            if r == 0:
                ax.set_title(county, fontsize=11, color=SDR_COLORS["ink"], pad=8)
            if c == 0:
                ax.set_ylabel(intervention, fontsize=10, color=SDR_COLORS["ink_soft"])

    fig.suptitle(
        f"{outcome_spec['summary_name']}: percent change from baseline by county",
        fontsize=16,
        y=1.01,
        color=SDR_COLORS["ink"],
    )

    from matplotlib.patches import Patch
    legend_handles = [
        Patch(facecolor=SCENARIO_COLORS[label], edgecolor="none", label=label)
        for label in CHANGE_LEVEL_LABELS
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        bbox_to_anchor=(0.5, -0.02),
        fontsize=10,
    )

    plt.tight_layout()

    timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M")
    summary_dir = PLOTS_ROOT / "summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    plot_name = f"{outcome_spec['plot_name']}_county_pct_change_subplots"
    png_path = summary_dir / f"{plot_name}_{timestamp}.png"
    csv_path = summary_dir / f"{plot_name}_{timestamp}.csv"

    # fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor=SDR_COLORS["paper"])
    grid_df.round(2).to_csv(csv_path, index=False)
    plt.show()
    # print(f"Saved pct-change subplots to: {png_path}")
    # print(f"Saved pct-change table to: {csv_path}")
    return grid_df


def _pct_change_diverging_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list(
        "sdr_pct_div", [SDR_COLORS["accent"], "#FFFFFF", SDR_COLORS["high"]]
    )


def _draw_pct_change_heatmap(ax, matrix, goodness, row_labels, col_keys, max_abs, value_fmt="{:.1f}%"):
    from matplotlib.colors import TwoSlopeNorm

    n_rows = len(row_labels)
    n_cols = len(col_keys)
    scenarios_per_county = len(CHANGE_LEVEL_LABELS)

    cmap = _pct_change_diverging_cmap()
    norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0.0, vmax=max_abs)

    im = ax.imshow(
        goodness,
        cmap=cmap,
        norm=norm,
        aspect="auto",
        interpolation="nearest",
    )

    for r in range(n_rows):
        for c in range(n_cols):
            value = matrix[r, c]
            if pd.notna(value):
                g_norm = goodness[r, c] / max_abs if max_abs else 0.0
                text_color = "white" if abs(g_norm) > 0.55 else SDR_COLORS["ink"]
                ax.text(
                    c, r, value_fmt.format(value),
                    ha="center", va="center",
                    fontsize=13, color=text_color,
                )

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels([scenario for (_, scenario) in col_keys], fontsize=11, color="black")
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(row_labels, fontsize=13, color="black")
    ax.tick_params(axis="both", colors="black")

    counties_in_order = []
    for county, _ in col_keys:
        if county not in counties_in_order:
            counties_in_order.append(county)
    for i, county in enumerate(counties_in_order):
        center = i * scenarios_per_county + (scenarios_per_county - 1) / 2
        ax.text(
            center, -0.75, county,
            ha="center", va="bottom",
            fontsize=13, fontweight="bold", color="black",
        )
        if i > 0:
            ax.axvline(i * scenarios_per_county - 0.5, color=SDR_COLORS["paper"], linewidth=3)

    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color=SDR_COLORS["paper"], linewidth=2, linestyle="-")
    ax.tick_params(which="minor", length=0)
    ax.tick_params(which="major", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_ylim(n_rows - 0.5, -1.4)
    return im


def _add_pct_change_colorbar(fig, ax, im, max_abs, lower_is_better=True, mixed_direction=False):
    cbar = fig.colorbar(
        im,
        ax=ax,
        orientation="horizontal",
        fraction=0.05,
        pad=0.22,
        aspect=36,
        shrink=0.92,
    )
    ticks = np.linspace(-max_abs, max_abs, 5)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(["Worsening", "", "No change", "", "Improvement"])
    cbar.ax.tick_params(labelsize=11, color="black", labelcolor="black")
    cbar.outline.set_visible(False)
    return cbar


def _save_summary_heatmap(fig, grid_df, plot_name):
    timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M")
    summary_dir = PLOTS_ROOT / "summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    png_path = summary_dir / f"{plot_name}_{timestamp}.png"
    csv_path = summary_dir / f"{plot_name}_{timestamp}.csv"
    # fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor=SDR_COLORS["paper"])
    grid_df.round(2).to_csv(csv_path, index=False)
    plt.show()
    # print(f"Saved pct-change heatmap to: {png_path}")
    # print(f"Saved pct-change table to: {csv_path}")


def plot_county_pct_change_heatmap(outcome_spec, exclude_interventions=None):
    grid_df = build_pct_change_grid(outcome_spec, exclude_interventions=exclude_interventions)
    if grid_df.empty:
        print(f"No data for {outcome_spec['summary_name']} pct-change heatmap")
        return grid_df

    interventions = [
        INTERVENTION_CONFIGS[k]["intervention_name"]
        for k, enabled in RUN_INTERVENTIONS.items()
        if enabled and INTERVENTION_CONFIGS[k]["intervention_name"] not in (exclude_interventions or [])
    ]

    col_keys = [(county, scenario) for county in COUNTIES for scenario in CHANGE_LEVEL_LABELS]
    n_rows = len(interventions)
    n_cols = len(col_keys)

    matrix = np.full((n_rows, n_cols), np.nan)
    for r, intervention in enumerate(interventions):
        for c, (county, scenario) in enumerate(col_keys):
            sel = grid_df[
                (grid_df["Intervention"] == intervention)
                & (grid_df["County"] == county)
                & (grid_df["Scenario Label"] == scenario)
            ]["Percent change vs baseline"]
            if not sel.empty:
                matrix[r, c] = sel.iloc[0]

    lower_is_better = outcome_spec.get("lower_is_better", True)
    goodness = (-matrix if lower_is_better else matrix)

    finite = np.abs(matrix[np.isfinite(matrix)])
    max_abs = max(finite.max(), 1.0) if finite.size else 1.0

    fig, ax = plt.subplots(figsize=(1.25 * n_cols + 2.8, 1.05 * n_rows + 2.6))
    fig.patch.set_facecolor(SDR_COLORS["paper"])

    im = _draw_pct_change_heatmap(ax, matrix, goodness, interventions, col_keys, max_abs)

    ax.set_title(
        f"{outcome_spec['summary_name']}: % change from baseline",
        fontsize=17, pad=34, color=SDR_COLORS["ink"],
    )
    _add_pct_change_colorbar(fig, ax, im, max_abs, lower_is_better=lower_is_better)

    plt.tight_layout(rect=(0, 0.04, 1, 1))
    _save_summary_heatmap(fig, grid_df, f"{outcome_spec['plot_name']}_county_pct_change_heatmap")
    return grid_df


def plot_prompts_extra_pct_change_heatmap(outcome_specs, intervention_name="PROMPTS", plot_name="prompts_extras"):
    rows_data = []
    for spec in outcome_specs:
        grid = build_pct_change_grid(spec)
        grid = grid[grid["Intervention"] == intervention_name].copy()
        if grid.empty:
            continue
        grid["Outcome"] = spec["summary_name"]
        grid["lower_is_better"] = spec.get("lower_is_better", True)
        rows_data.append(grid)

    if not rows_data:
        print("No data for PROMPTS extras pct-change heatmap")
        return pd.DataFrame()

    grid_df = pd.concat(rows_data, ignore_index=True)
    outcome_names = [s["summary_name"] for s in outcome_specs if s["summary_name"] in grid_df["Outcome"].unique()]

    col_keys = [(county, scenario) for county in COUNTIES for scenario in CHANGE_LEVEL_LABELS]
    n_rows = len(outcome_names)
    n_cols = len(col_keys)

    matrix = np.full((n_rows, n_cols), np.nan)
    goodness = np.full((n_rows, n_cols), np.nan)
    for r, outcome_name in enumerate(outcome_names):
        lower_is_better = bool(grid_df.loc[grid_df["Outcome"] == outcome_name, "lower_is_better"].iloc[0])
        for c, (county, scenario) in enumerate(col_keys):
            sel = grid_df[
                (grid_df["Outcome"] == outcome_name)
                & (grid_df["County"] == county)
                & (grid_df["Scenario Label"] == scenario)
            ]["Percent change vs baseline"]
            if not sel.empty:
                value = sel.iloc[0]
                matrix[r, c] = value
                goodness[r, c] = -value if lower_is_better else value

    finite = np.abs(matrix[np.isfinite(matrix)])
    max_abs = max(finite.max(), 1.0) if finite.size else 1.0

    fig, ax = plt.subplots(figsize=(1.25 * n_cols + 2.8, 1.05 * n_rows + 2.6))
    fig.patch.set_facecolor(SDR_COLORS["paper"])

    im = _draw_pct_change_heatmap(ax, matrix, goodness, outcome_names, col_keys, max_abs)

    ax.set_title(
        f"{intervention_name}: % change from baseline",
        fontsize=17, pad=34, color=SDR_COLORS["ink"],
    )
    _add_pct_change_colorbar(fig, ax, im, max_abs, mixed_direction=True)

    plt.tight_layout(rect=(0, 0.04, 1, 1))
    _save_summary_heatmap(fig, grid_df, f"{plot_name}_pct_change_heatmap")
    return grid_df


def run_intervention_plots(file_key, config):
    intervention_name = config["intervention_name"]
    outcomes_to_plot = config["outcomes_to_plot"]

    run_subset = load_intervention_run_data(file_key, intervention_name)
    loc_subset = load_intervention_location_data(file_key, intervention_name)

    timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M")
    file_prefix = intervention_name.replace(" ", "_").lower()
    plot_dir = PLOTS_ROOT / "county_comparison" / file_prefix

    print(f"\nRunning plots for: {intervention_name}")
    change_tables = []

    for outcome in outcomes_to_plot:
        run_summary = summarize_preaggregated_outcome(
            run_subset,
            outcome_col=outcome["outcome_col"],
            multiplier=outcome["multiplier"],
            outcome_name=outcome["summary_name"],
            group_cols=["County", "Intervention", "Scenario Label"],
        )

        plot_outcome_by_county(
            run_summary,
            outcome_name=outcome["summary_name"],
            ylabel=outcome["ylabel"],
            plot_name=f"{outcome['plot_name']}_county_comparison",
            intervention_name=intervention_name,
            file_prefix=file_prefix,
            plot_dir=plot_dir,
            timestamp=timestamp,
            x_axis="county" if file_key == "combined" else "scenario",
        )

        if outcome["outcome_col"] in loc_subset.columns:
            loc_summary = summarize_preaggregated_outcome(
                loc_subset,
                outcome_col=outcome["outcome_col"],
                multiplier=outcome["multiplier"],
                outcome_name=outcome["summary_name"],
                group_cols=["County", "Intervention", "Scenario Label", "Delivery Location"],
            )

            for county in COUNTIES:
                plot_outcome_by_location(
                    loc_summary,
                    outcome_name=outcome["summary_name"],
                    ylabel=outcome["ylabel"],
                    plot_name=f"{outcome['plot_name']}_location",
                    intervention_name=intervention_name,
                    county=county,
                    file_prefix=file_prefix,
                    plot_dir=plot_dir,
                    timestamp=timestamp,
                )
        else:
            # No location-level data for this outcome (e.g. normal referral).
            # Make a per-county overall bar plot with the 4 scenario bars instead.
            for county in COUNTIES:
                plot_outcome_single_county(
                    run_summary,
                    outcome_name=outcome["summary_name"],
                    ylabel=outcome["ylabel"],
                    plot_name=f"{outcome['plot_name']}_overall_by_county",
                    intervention_name=intervention_name,
                    county=county,
                    file_prefix=file_prefix,
                    plot_dir=plot_dir,
                    timestamp=timestamp,
                )

        change_tables.append(make_change_table(run_summary))

    for loc_plot in config.get("location_distribution_plots", []):
        l45_summary = summarize_preaggregated_outcome(
            run_subset,
            outcome_col=loc_plot["rate_col"],
            multiplier=100,
            outcome_name=loc_plot["plot_title"],
            group_cols=["County", "Intervention", "Scenario Label"],
        )
        plot_l45_by_county(
            l45_summary,
            plot_title=loc_plot["plot_title"],
            plot_name=loc_plot["plot_name"],
            intervention_name=intervention_name,
            file_prefix=file_prefix,
            plot_dir=plot_dir,
            timestamp=timestamp,
            x_axis="county" if file_key == "combined" else "scenario",
        )

        for county in COUNTIES:
            plot_outcome_single_county(
                l45_summary,
                outcome_name=loc_plot["plot_title"],
                ylabel="Share of mothers (%)",
                plot_name=f"{loc_plot['plot_name']}_overall_by_county",
                intervention_name=intervention_name,
                county=county,
                file_prefix=file_prefix,
                plot_dir=plot_dir,
                timestamp=timestamp,
            )

        change_tables.append(make_change_table(l45_summary))

    if change_tables:
        summary_table = pd.concat(change_tables, ignore_index=True)
        table_path = plot_dir / f"{file_prefix}_summary_change_table_{timestamp}.csv"
        # summary_table.to_csv(table_path, index=False)
        # print(f"Saved summary table to: {table_path}")
        display(summary_table)


In [8]:
# =========================================================
# RUN INTERVENTION PLOTS
# =========================================================

for file_key, should_run in RUN_INTERVENTIONS.items():
    if should_run:
        run_intervention_plots(file_key, INTERVENTION_CONFIGS[file_key])



Running plots for: MENTORS
Saved plot to: county_comparison/mentors/mentors_maternal_mortality_ratio_county_comparison_20260813_1450.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved plot to: county_comparison/mentors/kakamega/mentors_maternal_mortality_ratio_location_20260813_1450.png
Saved plot to: county_comparison/mentors/kisii/mentors_maternal_mortality_ratio_location_20260813_1450.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved plot to: county_comparison/mentors/mombasa/mentors_maternal_mortality_ratio_location_20260813_1450.png
Saved plot to: county_comparison/mentors/makueni/mentors_maternal_mortality_ratio_location_20260813_1450.png


...

Saved plot to: county_comparison/all/kakamega/all_initial_delivery_l45_rate_overall_by_county_20260813_1451.png
Saved plot to: county_comparison/all/kisii/all_initial_delivery_l45_rate_overall_by_county_20260813_1451.png
Saved plot to: county_comparison/all/mombasa/all_initial_delivery_l45_rate_overall_by_county_20260813_1451.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved plot to: county_comparison/all/makueni/all_initial_delivery_l45_rate_overall_by_county_20260813_1451.png
Saved plot to: county_comparison/all/all_final_delivery_l45_rate_20260813_1451.png
Saved plot to: county_comparison/all/kakamega/all_final_delivery_l45_rate_overall_by_county_20260813_1451.png
Saved plot to: county_comparison/all/kisii/all_final_delivery_l45_rate_overall_by_county_20260813_1451.png
Saved plot to: county_comparison/all/mombasa/all_final_delivery_l45_rate_overall_by_county_20260813_1451.png
Saved plot to: county_comparison/all/makueni/all_final_delivery_l45_rate_overall_by_county_20260813_1451.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,County,Intervention,Scenario Label,Outcome,mean,upper,lower,Absolute Change vs Baseline,Percent Change vs Baseline
0,Kakamega,All,Current,Maternal mortality ratio,213.90,224.20,203.61,-15.38,-6.71
1,Kakamega,All,High,Maternal mortality ratio,113.20,118.75,107.65,-116.09,-50.63
2,Kakamega,All,Moderate,Maternal mortality ratio,171.47,179.48,163.46,-57.81,-25.21
3,Kisii,All,Current,Maternal mortality ratio,54.63,57.09,52.17,-11.19,-17.00
4,Kisii,All,High,Maternal mortality ratio,23.11,24.17,22.05,-42.71,-64.89
...,...,...,...,...,...,...,...,...,...
67,Mombasa,All,High,Final delivery at L4/5,71.66,71.69,71.64,2.30,3.32
68,Mombasa,All,Moderate,Final delivery at L4/5,70.36,70.38,70.34,1.00,1.44
69,Makueni,All,Current,Final delivery at L4/5,65.95,65.98,65.92,0.24,0.36
70,Makueni,All,High,Final delivery at L4/5,72.50,72.50,72.50,6.79,10.33


In [9]:
# =========================================================
# COUNTY % CHANGE SUBPLOT GRIDS
# =========================================================

for outcome in BASE_OUTCOMES:
    exclude = PCT_CHANGE_EXCLUDE_INTERVENTIONS.get(outcome["summary_name"], [])
    grid = plot_county_pct_change_subplots(outcome, exclude_interventions=exclude)
    display(grid.head(12))

PROMPTS_EXTRA_OUTCOMES = [
    o for o in INTERVENTION_CONFIGS["prompts"]["outcomes_to_plot"]
    if o["summary_name"] not in {x["summary_name"] for x in BASE_OUTCOMES}
] + [
    {
        "summary_name": loc_plot["plot_title"],
        "outcome_col": loc_plot["rate_col"],
        "multiplier": 100,
        "plot_name": loc_plot["plot_name"],
        "lower_is_better": False,
    }
    for loc_plot in INTERVENTION_CONFIGS["prompts"].get("location_distribution_plots", [])
]

for outcome in PROMPTS_EXTRA_OUTCOMES:
    grid = build_pct_change_grid(outcome)
    grid = grid[grid["Intervention"] == "PROMPTS"]
    if grid.empty:
        continue

    fig, axes = plt.subplots(1, len(COUNTIES), figsize=(3.0 * len(COUNTIES) + 1.0, 3.8), sharey=True)
    fig.patch.set_facecolor(SDR_COLORS["paper"])
    if len(COUNTIES) == 1:
        axes = [axes]

    bar_colors = [SCENARIO_COLORS[l] for l in CHANGE_LEVEL_LABELS]
    y_limit = max(5, np.ceil(grid["Percent change vs baseline"].abs().max() / 5) * 5)

    for c, county in enumerate(COUNTIES):
        ax = axes[c]
        ax.set_facecolor(SDR_COLORS["card"])
        cell = grid[grid["County"] == county].set_index("Scenario Label").reindex(CHANGE_LEVEL_LABELS)
        values = cell["Percent change vs baseline"].to_numpy(dtype=float)
        x = np.arange(len(CHANGE_LEVEL_LABELS))
        ax.bar(x, values, color=bar_colors, width=0.62)
        for bar, value in zip(ax.patches, values):
            if pd.notna(value):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + np.sign(value) * 0.8,
                    f"{value:.1f}%",
                    ha="center",
                    va="bottom" if value >= 0 else "top",
                    fontsize=8,
                )
        ax.axhline(0, color=SDR_COLORS["ink_muted"], linewidth=0.8)
        ax.set_ylim(-y_limit, y_limit)
        ax.set_xticks(x)
        ax.set_xticklabels(CHANGE_LEVEL_LABELS, fontsize=8, rotation=35, ha="right")
        ax.set_title(county, fontsize=11)
        ax.grid(axis="y", alpha=0.4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        f"PROMPTS — {outcome['summary_name']}: percent change from baseline by county",
        fontsize=15,
        y=1.02,
    )
    plt.tight_layout()

    timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M")
    summary_dir = PLOTS_ROOT / "summary"
    summary_dir.mkdir(parents=True, exist_ok=True)
    plot_name = f"prompts_{outcome['plot_name']}_county_pct_change_subplots"
    png_path = summary_dir / f"{plot_name}_{timestamp}.png"
    csv_path = summary_dir / f"{plot_name}_{timestamp}.csv"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor=SDR_COLORS["paper"])
    grid.round(2).to_csv(csv_path, index=False)
    plt.show()
    print(f"Saved PROMPTS pct-change subplots to: {png_path}")


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/3163252819.py:157: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Intervention,County,Scenario Label,Percent change vs baseline
0,MENTORS,Kakamega,Current,-3.413420
1,MENTORS,Kakamega,Moderate,-5.483362
2,MENTORS,Kakamega,High,-7.293623
3,MENTORS,Kisii,Current,-9.743982
4,MENTORS,Kisii,Moderate,-13.060757
5,MENTORS,Kisii,High,-16.713794
6,MENTORS,Mombasa,Current,-7.266764
7,MENTORS,Mombasa,Moderate,-10.822473
8,MENTORS,Mombasa,High,-14.145163
9,MENTORS,Makueni,Current,-9.709047


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/3163252819.py:157: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Intervention,County,Scenario Label,Percent change vs baseline
0,MENTORS,Kakamega,Current,-6.943072
1,MENTORS,Kakamega,Moderate,-11.726108
2,MENTORS,Kakamega,High,-15.885298
3,MENTORS,Kisii,Current,-15.449506
4,MENTORS,Kisii,Moderate,-20.731268
5,MENTORS,Kisii,High,-26.132048
6,MENTORS,Mombasa,Current,-13.309721
7,MENTORS,Mombasa,Moderate,-18.336122
8,MENTORS,Mombasa,High,-23.212875
9,MENTORS,Makueni,Current,-16.739407


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/1650466808.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved PROMPTS pct-change subplots to: summary/prompts_antenatal_care_rate_county_pct_change_subplots_20260813_1451.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/1650466808.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/1650466808.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved PROMPTS pct-change subplots to: summary/prompts_normal_referral_county_pct_change_subplots_20260813_1451.png
Saved PROMPTS pct-change subplots to: summary/prompts_initial_delivery_l45_rate_county_pct_change_subplots_20260813_1451.png
Saved PROMPTS pct-change subplots to: summary/prompts_final_delivery_l45_rate_county_pct_change_subplots_20260813_1451.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/1650466808.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# =========================================================
# COUNTY % CHANGE HEATMAPS (compact alternative to bar subplots)
# =========================================================

for outcome in BASE_OUTCOMES:
    exclude = PCT_CHANGE_EXCLUDE_INTERVENTIONS.get(outcome["summary_name"], [])
    plot_county_pct_change_heatmap(outcome, exclude_interventions=exclude)

plot_prompts_extra_pct_change_heatmap(PROMPTS_EXTRA_OUTCOMES)


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/3163252819.py:258: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/3163252819.py:258: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/3163252819.py:258: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Intervention,County,Scenario Label,Percent change vs baseline,Outcome,lower_is_better
0,PROMPTS,Kakamega,Current,0.000000,Antenatal care rate,False
1,PROMPTS,Kakamega,Moderate,9.014359,Antenatal care rate,False
2,PROMPTS,Kakamega,High,33.303828,Antenatal care rate,False
3,PROMPTS,Kisii,Current,0.294686,Antenatal care rate,False
4,PROMPTS,Kisii,Moderate,8.954439,Antenatal care rate,False
5,PROMPTS,Kisii,High,33.399095,Antenatal care rate,False
6,PROMPTS,Mombasa,Current,0.364256,Antenatal care rate,False
7,PROMPTS,Mombasa,Moderate,9.066403,Antenatal care rate,False
8,PROMPTS,Mombasa,High,33.439467,Antenatal care rate,False
9,PROMPTS,Makueni,Current,0.228116,Antenatal care rate,False


In [11]:
# =========================================================
# VALIDATION
# =========================================================

checks = [
    ("Kakamega", "mentors", "mentors_current", "maternal_deaths_per_100k", 1),
    ("Kisii", "mentors", "mentors_current", "maternal_deaths_per_100k", 1),
    ("Mombasa", "fqa", "fqa_high", "maternal_deaths_per_100k", 1),
]

validation_rows = []
for county, file_key, scenario, col, multiplier in checks:
    mean_df = pd.read_csv(resolve_county_csv(COUNTY_DIRS[county], "scenario_mean_results.csv"))
    intervention_name = INTERVENTION_CONFIGS[file_key]["intervention_name"]
    subset = load_intervention_run_data(file_key, intervention_name)
    subset = subset[(subset["County"] == county) & (subset["scenario"] == scenario)]

    notebook_mean = (subset[col] * multiplier).mean()
    reference_mean = mean_df.loc[mean_df["scenario"] == scenario, col].iloc[0] * multiplier

    validation_rows.append({
        "county": county,
        "scenario": scenario,
        "notebook_mean": round(notebook_mean, 4),
        "scenario_mean_results": round(reference_mean, 4),
        "difference": round(notebook_mean - reference_mean, 6),
    })

validation_table = pd.DataFrame(validation_rows)
display(validation_table)

if (validation_table["difference"].abs() > 1e-6).any():
    print("Warning: notebook means differ from scenario_mean_results.csv")
else:
    print("Validation passed")


,county,scenario,notebook_mean,scenario_mean_results,difference
0,Kakamega,mentors_current,221.4595,221.4595,0.0
1,Kisii,mentors_current,59.4088,59.4088,-0.0
2,Mombasa,fqa_high,55.5143,55.5143,0.0


Validation passed


In [12]:
# =========================================================
# FQA DIAGNOSTIC: ONE COUNTY / ONE INTERVENTION FAMILY
# =========================================================
# Choose one county, then set RUN_FQA_DIAGNOSTIC = True to regenerate this
# diagnostic batch. Results are saved separately from scenario_comparison.py.

import scenario_comparison as sc
import parameter_loader
from global_func import reset_E, reset_HSS, reset_S, reset_flags
from model_run import run_model_dash
from parameter_loader import calculate_derived_parameters, get_parameters, get_slider_params

FQA_DIAGNOSTIC_COUNTY = "Kisii"  # choose: "Kakamega", "Kisii", "Mombasa", "Makueni"
RUN_FQA_DIAGNOSTIC = True
PLOT_FQA_DIAGNOSTIC = True

FQA_DIAGNOSTIC_SKIP_INITIAL_RUNS = 200
FQA_DIAGNOSTIC_ANALYSIS_RUNS = 200
FQA_DIAGNOSTIC_FIRST_ANALYSIS_RUN = FQA_DIAGNOSTIC_SKIP_INITIAL_RUNS + 1
FQA_DIAGNOSTIC_N_RUNS = FQA_DIAGNOSTIC_SKIP_INITIAL_RUNS + FQA_DIAGNOSTIC_ANALYSIS_RUNS

FQA_DIAGNOSTIC_SCENARIOS = [
    {"scenario": "fqa_base", "intervention": "FQA", "level": "base",
     "description": "PULSE current; FQA off; all single interventions on"},
    {"scenario": "fqa_current", "intervention": "FQA", "level": "current",
     "description": "County-specific current FQA index; PULSE current; all single interventions on; 10% amplification"},
    {"scenario": "fqa_moderate", "intervention": "FQA", "level": "moderate",
     "description": "50% FQA implementation index; PULSE current; all single interventions on; 10% amplification"},
    {"scenario": "fqa_high", "intervention": "FQA", "level": "high",
     "description": "95% FQA implementation index; PULSE current; all single interventions on; 30% amplification"},
]

FQA_DIAGNOSTIC_OUTCOMES = [
    {
        "summary_name": "Maternal mortality ratio",
        "outcome_col": "maternal_deaths_per_100k",
        "multiplier": 1,
        "ylabel": "MMR per 100,000 live births",
        "plot_name": "maternal_mortality_ratio",
    },
    {
        "summary_name": "Maternal complications",
        "outcome_col": "severe_complication",
        "multiplier": 100,
        "ylabel": "Maternal complications (%)",
        "plot_name": "maternal_complications",
    },
]


def _fqa_diagnostic_county_slug(county):
    if county not in COUNTY_FOLDER_SLUGS:
        raise ValueError(f"Choose one of {list(COUNTY_FOLDER_SLUGS)}; got {county!r}")
    return COUNTY_FOLDER_SLUGS[county]


def fqa_diagnostic_output_dir(county=FQA_DIAGNOSTIC_COUNTY):
    slug = _fqa_diagnostic_county_slug(county)
    return DATA_ROOT / f"fqa_diagnostic_results_{slug}"


def run_fqa_diagnostic(county=FQA_DIAGNOSTIC_COUNTY):
    county_slug = _fqa_diagnostic_county_slug(county)
    output_dir = fqa_diagnostic_output_dir(county)
    output_dir.mkdir(parents=True, exist_ok=True)

    parameter_loader.WORKBOOK_PATH = sc.WORKBOOK_PATH
    slider_params = get_slider_params(county=county_slug)

    rows = []
    location_rows = []
    run_indices = range(FQA_DIAGNOSTIC_FIRST_ANALYSIS_RUN - 1, FQA_DIAGNOSTIC_N_RUNS)

    for completed_runs, run in enumerate(run_indices, start=1):
        parameter_seed = sc.BASE_SEED + run
        month_seeds = np.arange(parameter_seed, parameter_seed + sc.N_MONTHS, dtype=int)

        for scenario_def in FQA_DIAGNOSTIC_SCENARIOS:
            scenario = scenario_def["scenario"]

            param = get_parameters(county=county_slug, seed=parameter_seed)
            param = calculate_derived_parameters(param)
            param.update({
                "E": reset_E(),
                "S": reset_S(slider_params),
                "HSS": reset_HSS(slider_params),
            })

            flags = reset_flags()
            sc.configure_scenario(scenario, param, flags, slider_params)

            _, individuals, _ = run_model_dash(
                param,
                flags,
                n_months=sc.N_MONTHS,
                int_period=sc.N_MONTHS,
                base_seed=month_seeds,
            )

            rows.append({
                "scenario": scenario,
                "intervention": scenario_def["intervention"],
                "level": scenario_def["level"],
                "run": run + 1,
                **sc.summarize(individuals),
            })

            for location_result in sc.summarize_by_location(individuals):
                location_rows.append({
                    "scenario": scenario,
                    "intervention": scenario_def["intervention"],
                    "level": scenario_def["level"],
                    "run": run + 1,
                    **location_result,
                })

        if completed_runs % 10 == 0 or completed_runs == 1:
            print(
                f"{county}: completed {completed_runs}/{FQA_DIAGNOSTIC_ANALYSIS_RUNS} "
                f"diagnostic runs using run {run + 1}/{FQA_DIAGNOSTIC_N_RUNS}",
                flush=True,
            )

    scenario_definitions = pd.DataFrame(FQA_DIAGNOSTIC_SCENARIOS)
    run_results = pd.DataFrame(rows)
    location_run_results = pd.DataFrame(location_rows)

    metric_columns = [
        c for c in run_results.columns
        if c not in {"scenario", "intervention", "level", "run"}
    ]
    mean_results = (
        run_results
        .groupby(["scenario", "intervention", "level"], sort=False)[metric_columns]
        .mean()
        .reset_index()
    )

    location_keys = ["scenario", "intervention", "level", "location", "location_code"]
    location_metric_columns = [
        c for c in location_run_results.columns
        if c not in {*location_keys, "run"}
    ]
    location_mean_results = (
        location_run_results
        .groupby(location_keys, sort=False)[location_metric_columns]
        .mean()
        .reset_index()
    )

    baseline = mean_results.loc[
        mean_results["scenario"] == "fqa_base", metric_columns
    ].iloc[0]
    differences_from_fqa_base = mean_results.copy()
    differences_from_fqa_base[metric_columns] = (
        mean_results[metric_columns].to_numpy() - baseline.to_numpy()
    )
    differences_from_fqa_base.insert(3, "comparison_baseline", "fqa_base")

    baseline_by_location = location_mean_results.loc[
        location_mean_results["scenario"] == "fqa_base",
        ["location_code", *location_metric_columns],
    ].set_index("location_code")

    location_differences_from_fqa_base = location_mean_results.copy()
    for index, row in location_differences_from_fqa_base.iterrows():
        baseline_location = baseline_by_location.loc[row["location_code"]]
        location_differences_from_fqa_base.loc[index, location_metric_columns] = (
            row[location_metric_columns].to_numpy(dtype=float)
            - baseline_location.to_numpy(dtype=float)
        )
    location_differences_from_fqa_base.insert(3, "comparison_baseline", "fqa_base")

    scenario_definitions.to_csv(output_dir / "scenario_definitions.csv", index=False)
    run_results.to_csv(output_dir / "scenario_run_results.csv", index=False)
    location_run_results.to_csv(output_dir / "scenario_location_run_results.csv", index=False)
    mean_results.to_csv(output_dir / "scenario_mean_results.csv", index=False)
    location_mean_results.to_csv(output_dir / "scenario_location_mean_results.csv", index=False)
    differences_from_fqa_base.to_csv(output_dir / "scenario_differences_from_fqa_base.csv", index=False)
    location_differences_from_fqa_base.to_csv(
        output_dir / "scenario_location_differences_from_fqa_base.csv",
        index=False,
    )

    print(f"Saved FQA diagnostic outputs to: {output_dir}")

    return {
        "scenario_definitions": scenario_definitions,
        "run_results": run_results,
        "location_run_results": location_run_results,
        "mean_results": mean_results,
        "location_mean_results": location_mean_results,
        "differences_from_fqa_base": differences_from_fqa_base,
        "location_differences_from_fqa_base": location_differences_from_fqa_base,
    }


def load_fqa_diagnostic_results(county=FQA_DIAGNOSTIC_COUNTY):
    output_dir = fqa_diagnostic_output_dir(county)
    required = {
        "scenario_definitions": output_dir / "scenario_definitions.csv",
        "run_results": output_dir / "scenario_run_results.csv",
        "location_run_results": output_dir / "scenario_location_run_results.csv",
        "mean_results": output_dir / "scenario_mean_results.csv",
        "location_mean_results": output_dir / "scenario_location_mean_results.csv",
        "differences_from_fqa_base": output_dir / "scenario_differences_from_fqa_base.csv",
        "location_differences_from_fqa_base": output_dir / "scenario_location_differences_from_fqa_base.csv",
    }

    missing = [path for path in required.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "No saved FQA diagnostic results found. "
            "Set RUN_FQA_DIAGNOSTIC = True and rerun this cell. "
            f"Missing: {missing[0]}"
        )

    return {name: pd.read_csv(path) for name, path in required.items()}


def _prepare_fqa_diagnostic_run_data(results, county=FQA_DIAGNOSTIC_COUNTY):
    df = results["run_results"].copy()
    df["County"] = county
    df["Scenario Label"] = df["level"].map(LEVEL_TO_LABEL)
    df["Intervention"] = "FQA"
    df["Run"] = df["run"]
    return df


def _prepare_fqa_diagnostic_location_data(results, county=FQA_DIAGNOSTIC_COUNTY):
    df = results["location_run_results"].copy()
    df["County"] = county
    df["Scenario Label"] = df["level"].map(LEVEL_TO_LABEL)
    df["Intervention"] = "FQA"
    df["Run"] = df["run"]
    df["Delivery Location"] = df["location"].astype(str).str.strip()
    return df


def plot_fqa_diagnostic_outcome(results, outcome_spec, county=FQA_DIAGNOSTIC_COUNTY):
    diagnostic_df = _prepare_fqa_diagnostic_run_data(results, county)
    summary = summarize_preaggregated_outcome(
        diagnostic_df,
        outcome_spec["outcome_col"],
        outcome_spec["multiplier"],
        outcome_spec["summary_name"],
        ["County", "Intervention", "Scenario Label"],
    )

    plot_df = (
        summary[summary["County"] == county]
        .set_index("Scenario Label")
        .reindex(SCENARIOS)
    )
    plot_df = plot_df[pd.notna(plot_df["mean"])]

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    fig.patch.set_facecolor(SDR_COLORS["paper"])
    ax.set_facecolor(SDR_COLORS["card"])

    x = np.arange(len(plot_df))
    bars = ax.bar(
        x,
        plot_df["mean"],
        yerr=plot_df["ci95"],
        capsize=2,
        color=[SCENARIO_COLORS[label] for label in plot_df.index],
        edgecolor="none",
        alpha=0.95,
        error_kw={"elinewidth": 1, "ecolor": SDR_COLORS["ink_soft"], "capthick": 1},
    )
    _annotate_bars(ax, bars, plot_df["mean"])

    ax.set_title(
        f"FQA diagnostic: {outcome_spec['summary_name']} — {county}",
        fontsize=17,
        pad=14,
        color=SDR_COLORS["ink"],
    )
    ax.set_ylabel(outcome_spec["ylabel"], fontsize=11)
    ax.set_xlabel("Scenario", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, fontsize=10)
    _style_axes(ax)
    ax.margins(y=0.22)
    plt.tight_layout()

    plot_dir = fqa_diagnostic_output_dir(county) / "plots"
    timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M")
    _save_plot(
        fig,
        plot_dir,
        "fqa_diagnostic",
        f"overall_{outcome_spec['plot_name']}",
        timestamp,
    )

    return summary


def plot_fqa_diagnostic_location_outcome(results, outcome_spec, county=FQA_DIAGNOSTIC_COUNTY):
    diagnostic_location_df = _prepare_fqa_diagnostic_location_data(results, county)
    summary = summarize_preaggregated_outcome(
        diagnostic_location_df,
        outcome_spec["outcome_col"],
        outcome_spec["multiplier"],
        outcome_spec["summary_name"],
        ["County", "Intervention", "Delivery Location", "Scenario Label"],
    )

    plot_dir = fqa_diagnostic_output_dir(county) / "plots"
    timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M")
    plot_outcome_by_location(
        summary,
        outcome_spec["summary_name"],
        outcome_spec["ylabel"],
        f"fqa_diagnostic_location_{outcome_spec['plot_name']}",
        "FQA diagnostic",
        county,
        "fqa_diagnostic",
        plot_dir,
        timestamp,
    )

    return summary


if RUN_FQA_DIAGNOSTIC:
    fqa_diagnostic_results = run_fqa_diagnostic(FQA_DIAGNOSTIC_COUNTY)
else:
    fqa_diagnostic_results = load_fqa_diagnostic_results(FQA_DIAGNOSTIC_COUNTY)
    print(f"Loaded saved FQA diagnostic outputs from: {fqa_diagnostic_output_dir(FQA_DIAGNOSTIC_COUNTY)}")

if PLOT_FQA_DIAGNOSTIC and fqa_diagnostic_results is not None:
    for outcome in FQA_DIAGNOSTIC_OUTCOMES:
        display(plot_fqa_diagnostic_outcome(fqa_diagnostic_results, outcome, FQA_DIAGNOSTIC_COUNTY))
        display(plot_fqa_diagnostic_location_outcome(fqa_diagnostic_results, outcome, FQA_DIAGNOSTIC_COUNTY))

Kisii: completed 1/200 diagnostic runs using run 201/400
Kisii: completed 10/200 diagnostic runs using run 210/400
Kisii: completed 20/200 diagnostic runs using run 220/400
Kisii: completed 30/200 diagnostic runs using run 230/400
Kisii: completed 40/200 diagnostic runs using run 240/400


/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)
/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)
/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)
/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)


Kisii: completed 50/200 diagnostic runs using run 250/400
Kisii: completed 60/200 diagnostic runs using run 260/400
Kisii: completed 70/200 diagnostic runs using run 270/400
Kisii: completed 80/200 diagnostic runs using run 280/400
Kisii: completed 90/200 diagnostic runs using run 290/400
Kisii: completed 100/200 diagnostic runs using run 300/400
Kisii: completed 110/200 diagnostic runs using run 310/400
Kisii: completed 120/200 diagnostic runs using run 320/400
Kisii: completed 130/200 diagnostic runs using run 330/400


/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)
/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)
/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)
/Users/poppy/Documents/GitHub/SDR-Dashboard-2026-main/global_func.py:21: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  solution = fsolve(equations, initial_guess)


Kisii: completed 140/200 diagnostic runs using run 340/400
Kisii: completed 150/200 diagnostic runs using run 350/400
Kisii: completed 160/200 diagnostic runs using run 360/400
Kisii: completed 170/200 diagnostic runs using run 370/400
Kisii: completed 180/200 diagnostic runs using run 380/400
Kisii: completed 190/200 diagnostic runs using run 390/400
Kisii: completed 200/200 diagnostic runs using run 400/400
Saved FQA diagnostic outputs to: fqa_diagnostic_results_kisii
Saved plot to: fqa_diagnostic_results_kisii/plots/fqa_diagnostic_overall_maternal_mortality_ratio_20260813_1457.png


...

/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,County,Intervention,Scenario Label,mean,sd,n_runs,ci95,lower,upper,Outcome
0,Kisii,FQA,Baseline,3.473309,1.241516,200,0.172065,3.301243,3.645374,Maternal complications
1,Kisii,FQA,Current,3.472519,1.241267,200,0.172031,3.300488,3.644550,Maternal complications
2,Kisii,FQA,High,3.410791,1.233725,200,0.170986,3.239806,3.581777,Maternal complications
3,Kisii,FQA,Moderate,3.441648,1.237546,200,0.171515,3.270132,3.613163,Maternal complications


Saved plot to: fqa_diagnostic_results_kisii/plots/kisii/fqa_diagnostic_fqa_diagnostic_location_maternal_complications_20260813_1457.png


/var/folders/70/863t0mz55rbbjfq24prwn94m0000gn/T/ipykernel_30249/191103605.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,County,Intervention,Delivery Location,Scenario Label,mean,sd,n_runs,ci95,lower,upper,Outcome
0,Kisii,FQA,Home,Baseline,3.712427,1.257187,200,0.174237,3.538190,3.886664,Maternal complications
1,Kisii,FQA,Home,Current,3.712937,1.257597,200,0.174294,3.538643,3.887231,Maternal complications
2,Kisii,FQA,Home,High,3.734873,1.261071,200,0.174775,3.560097,3.909648,Maternal complications
3,Kisii,FQA,Home,Moderate,3.707422,1.255849,200,0.174052,3.533371,3.881474,Maternal complications
4,Kisii,FQA,L2/3,Baseline,3.599867,1.186825,200,0.164486,3.435381,3.764352,Maternal complications
5,Kisii,FQA,L2/3,Current,3.599839,1.186689,200,0.164467,3.435372,3.764305,Maternal complications
6,Kisii,FQA,L2/3,High,3.548003,1.202043,200,0.166595,3.381409,3.714598,Maternal complications
7,Kisii,FQA,L2/3,Moderate,3.579627,1.196125,200,0.165774,3.413853,3.745402,Maternal complications
8,Kisii,FQA,L4,Baseline,1.927914,0.736408,200,0.102061,1.825853,2.029975,Maternal complications
9,Kisii,FQA,L4,Current,1.927101,0.736073,200,0.102015,1.825087,2.029116,Maternal complications
